# 03 — Longitudinal merge

**Input:** `Data/`, `Dictionary/`, `2018 Webscraping/`
**Output:** `Results/3.xlsx`

Merges the nine TYNDP editions into a single investment-level panel with
two-level columns (year × variable). The 2014–2026 cycles are linked by
Investment ID; the 2010–2013 cycles, which predate that identifier, are linked
by substation pair, with known parallel investments at identical nodes split
explicitly.

Investments that disappear from a report without an explicit status update are
assigned a final status from their expected commissioning year: commissioned if
it falls before the next report, cancelled otherwise.

In [1]:
import pandas as pd
import numpy as np
import json
import os
import ast
import copy
import re
import pycountry
from thefuzz import fuzz, process
from functools import reduce
import matplotlib.pyplot as plt
from collections import Counter
from collections import defaultdict
import plotly.express as px
import plotly.graph_objects as go


In [2]:
# Import script-functions
from scripts.config import TYNDP_DATA_PATHS, SHEET_NAMES, SKIP_ROWS
from scripts.data_cleaning import clean_project_data
from scripts.names_mismatch_2012 import fix_wrong_names

In [3]:
# ---Load Dictionary ---

with open('Dictionary/column_renames.json', 'r') as json_file:
    column_renames = json.load(json_file)
    
with open('Dictionary/status_mapping.json', 'r') as json_file:
    status_mapping = json.load(json_file)
    
with open('Dictionary/element_mapping.json', 'r') as json_file:
    element_mapping = json.load(json_file)

with open('Dictionary/crossborder_code.json', 'r') as json_file:
    crossborder_code = json.load(json_file)

In [4]:
# --- Load Data ---
tyndp_data = {}

for year in TYNDP_DATA_PATHS.keys():
    tyndp_data[year] = pd.read_excel(
        TYNDP_DATA_PATHS[year], 
        sheet_name=SHEET_NAMES[year], 
        skiprows=SKIP_ROWS[year]
    )

    # Below 339 are reported storage investments
    if year == '2018':
        tyndp_data[year] = tyndp_data[year].iloc[:339]


In [5]:
df = copy.deepcopy(tyndp_data)

### 2018 status from web scraping

In [6]:
# Load 2018 data from web scaping
df_new = pd.read_csv("2018 Webscraping/tyndp2018_project_investments.csv")

df['2018'] = df['2018'].merge(
    df_new,
    left_on="Investment ID",
    right_on="investment_id",
    how="left"
)

df['2018'].drop(columns=["investment_id"], inplace=True, errors='ignore')
df['2018'].drop(columns=["project_url"], inplace=True, errors='ignore')


### Column renaming and data cleaning in the original dataset

In [7]:
for year in df:
    df[year].columns = df[year].columns.str.strip().str.replace(r"\s+", " ", regex=True)

for year, mapping in column_renames.items():
    df[year].rename(columns=mapping, inplace=True)

In [8]:
df['2013']['Project_ID'] = df['2013']['Inv_index']


In [9]:
df = clean_project_data(df, column_renames)

2010: 495 rows before, 495 rows after 'Project ID' cleanup
2012: 515 rows before, 515 rows after 'Project ID' cleanup
2012: 515 rows before , 515 rows after 'Investment index' cleanup
2013: 755 rows before, 755 rows after 'Project ID' cleanup
2013: 755 rows before , 485 rows after 'Investment index' cleanup
2014: 498 rows before, 498 rows after 'Project ID' cleanup
2014: 498 rows before , 371 rows after 'Investment index' cleanup
2015: 422 rows before , 422 rows after 'Investment index' cleanup
2016: 611 rows before, 611 rows after 'Project ID' cleanup
2016: 611 rows before , 434 rows after 'Investment index' cleanup
2018: 339 rows before, 339 rows after 'Project ID' cleanup
2018: 339 rows before , 337 rows after 'Investment index' cleanup
2020_projects: 641 rows before, 154 rows after 'Project ID' cleanup
2020_invest: 321 rows before, 321 rows after 'Project ID' cleanup
2020_invest: 321 rows before , 321 rows after 'Investment index' cleanup
2022_projects: 641 rows before, 141 rows af

### Status mapping
Under consideration = 1 \
In Planning but not permitting = 2 \
Design & permitting = 3 \
Under construction = 4 \
Commissioned = 5 \
Cancelled = 6 \
nan = 7

In [10]:
for year in ['2010','2012','2013','2014','2015','2016', '2018', '2026_invest']:
    df[year]['Inv_Status'] = df[year]['Inv_Status'].map(status_mapping)


### Create a 2-level (hierarchical) column index:
Top level = the year, Bottom level = original column name.

In [11]:
for year in df:
    old_cols = df[year].columns
    df[year].columns = pd.MultiIndex.from_tuples(
        [(year, col) for col in old_cols],
        names=["Year", "Column"] 
    )

In [12]:
df['2015'][('2015', 'Inv_Substation From')] = df['2015'][('2015', 'Inv_Substation From')].str.strip()
df['2015'] = df['2015'][df['2015'][('2015', 'Inv_Substation From')] != 'Substation 1']

Inv_index code for 15 inv in 2012 where different from the one in 2013 and have been manually checked and corrected

In [13]:
df = fix_wrong_names(df)

### Cross border 2016 2026

In [14]:
# --- 2016 ---
df['2016'][('2016', 'Project_Is_cross_border')] = (
    df['2016'][('2016', 'Project_Is_cross_border')]
    .map(crossborder_code)
    .fillna(0)
    .astype(int)
)

# --- 2026 ---
df['2026_projects'][('2026_projects', 'Project_Is_cross_border')] = (
    df['2026_projects'][('2026_projects', 'Project_Is_cross_border')]
    .map(crossborder_code)
    .fillna(0)
    .astype(int)
)

### Merging 2020 22 24 - project and investment sheet together

In [15]:
for year in ['2020', '2022', '2024', '2026']:
    invest_key = f'{year}_invest'
    proj_key   = f'{year}_projects'

    df[invest_key].columns = df[invest_key].columns.map(
        lambda col: col[1] if isinstance(col, tuple) else col
    )
    df[proj_key].columns = df[proj_key].columns.map(
        lambda col: col[1] if isinstance(col, tuple) else col
    )

    df[invest_key]["Project_ID"] = (
        df[invest_key]["Project_ID"]
        .astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    )
    df[proj_key]["Project_ID"] = (
        df[proj_key]["Project_ID"]
        .astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    )

    merged_df = pd.merge(
        df[invest_key],
        df[proj_key],
        on="Project_ID",
        how="outer",
        suffixes=("_invest", "_projects")
    )

    merged_df.columns = pd.MultiIndex.from_arrays(
        [[year]*len(merged_df.columns), merged_df.columns],
        names=["Year", "Column"]
    )

    df[year] = merged_df


### Merging 2010 2013

In [16]:
# Merge 2012 2013 on 'Investment index'
years_to_merge = ['2012', '2013']

dfs = [df[year] for year in years_to_merge]

merged_2012_2013 = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        left_on=[(left.columns.levels[0][0], 'Inv_index')],
        right_on=[(right.columns.levels[0][0], 'Inv_index')],
        how='outer'
    ),
    dfs
)

df['merged_2012_2013'] = merged_2012_2013

In [17]:
cols_subset = [('2010', 'Inv_Substation From'), ('2010', 'Inv_Substation To')]
df['2010'] = df['2010'].drop_duplicates(subset=cols_subset, keep='first')

In [18]:
df_12_13 = df['merged_2012_2013']
df_2010   = df['2010']

merged = pd.merge(
    df_12_13,
    df_2010,
    how='left',
    left_on=[('2012', 'Inv_Substation From'), ('2012', 'Inv_Substation To')],
    right_on=[('2010', 'Inv_Substation From'), ('2010', 'Inv_Substation To')]
)

idx = pd.IndexSlice
merged_2010_2012_2013 = merged.loc[:, idx[['2010', '2012', '2013'], :]]

df['merged_2010_2012_2013'] = merged_2010_2012_2013

### Create the two column inv substation in 2026 dataset with non empty datata

In [19]:
df['2026'][('2026', 'Inv_Substation From')] = 'XX'
df['2026'][('2026', 'Inv_Substation To')] = 'XX'

### Merge 2014-2026 

In [20]:
for year in ['2014','2015','2016', '2018', '2020', '2022', '2024', '2026']:
    df[year] = df[year].loc[:, ~df[year].columns.duplicated()]

In [21]:
years_to_merge = ['2014', '2015','2016', '2018', '2020', '2022', '2024', '2026']

dfs = [df[year] for year in years_to_merge]

merged_2014_2024 = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        left_on=[(left.columns.levels[0][0], 'Inv_index')],
        right_on=[(right.columns.levels[0][0], 'Inv_index')],
        how='outer'
    ),
    dfs
)

df['merged_2014_2024'] = merged_2014_2024

In [22]:
if ('2014', 'Inv_index') in merged_2014_2024.columns:
    merged_2014_2024[('meta', 'Inv_index')] = merged_2014_2024[('2014', 'Inv_index')]

    mask_no_proj = merged_2014_2024[('2014', 'Project_ID')].isnull() | \
                   (merged_2014_2024[('2014', 'Project_ID')].astype(str).str.strip() == "")
    merged_2014_2024.loc[mask_no_proj, ('2014', 'Inv_index')] = np.nan

### Merging 2010_13 with 14_24
Keeping project from 2010_13 only if present in 2014

In [23]:
mask_13_east = (df['merged_2010_2012_2013'][('2013', 'Inv_Substation From')] == "Under Consideration (GB)") & \
               (df['merged_2010_2012_2013'][('2013', 'Inv_Description')].str.contains("East Coast", na=False))
mask_13_wind = (df['merged_2010_2012_2013'][('2013', 'Inv_Substation From')] == "Under Consideration (GB)") & \
               (df['merged_2010_2012_2013'][('2013', 'Inv_Description')].str.contains("Wind", na=False))

df['merged_2010_2012_2013'].loc[mask_13_east, [('2013', 'Inv_Substation From'), ('2013', 'Inv_Substation To')]] = "Under Consideration (GB) 1"
df['merged_2010_2012_2013'].loc[mask_13_wind, [('2013', 'Inv_Substation From'), ('2013', 'Inv_Substation To')]] = "Under Consideration (GB) 2"

mask_14_east = (df['merged_2014_2024'][('2014', 'Inv_Substation From')] == "Under Consideration (GB)") & \
               (df['merged_2014_2024'][('2014', 'Inv_Description')].str.contains("East Coast", na=False))
mask_14_wind = (df['merged_2014_2024'][('2014', 'Inv_Substation From')] == "Under Consideration (GB)") & \
               (df['merged_2014_2024'][('2014', 'Inv_Description')].str.contains("Wind", na=False))

df['merged_2014_2024'].loc[mask_14_east, [('2014', 'Inv_Substation From'), ('2014', 'Inv_Substation To')]] = "Under Consideration (GB) 1"
df['merged_2014_2024'].loc[mask_14_wind, [('2014', 'Inv_Substation From'), ('2014', 'Inv_Substation To')]] = "Under Consideration (GB) 2"


mask_krajnik_pst = (df['merged_2010_2012_2013'][('2013', 'Inv_index')] == "94. A70")
df['merged_2010_2012_2013'].loc[mask_krajnik_pst, ('2013', 'Inv_Substation From')] = "Krajnik (PL) - PST Only (No Match)"

mask_mikulowa_upgrade = (df['merged_2010_2012_2013'][('2013', 'Inv_index')] == "94. A69")
df['merged_2010_2012_2013'].loc[mask_mikulowa_upgrade, ('2013', 'Inv_Substation From')] = "Mikulowa (PL) - Upgrade Only (No Match)"

In [24]:
df_10_12_13 = df['merged_2010_2012_2013']
df_14_24     = df['merged_2014_2024']

df_10_12_13 = df_10_12_13.replace({
    ('2013','Inv_Substation From'): {'': pd.NA},
    ('2013','Inv_Substation To'):   {'': pd.NA}
})

df_10_clean = df_10_12_13.dropna(subset=[
    ('2013','Inv_Substation From'),
    ('2013','Inv_Substation To')
])

merged_2010_2024 = pd.merge(
    df_14_24,
    df_10_clean,
    how='left',
    left_on =[('2014','Inv_Substation From'), ('2014','Inv_Substation To')],
    right_on=[('2013','Inv_Substation From'), ('2013','Inv_Substation To')]
)

idx = pd.IndexSlice
merged_2010_2024 = merged_2010_2024.loc[:, idx[
    ['2010','2012','2013','2014','2015','2016','2018','2020','2022','2024','2026','meta'],
    :
]]


### Standardize commissioning year
In the initial dataset there are errors, like for example 00-0000, as expected commissioning year, so the code filter this years and add a na



In [25]:
years_all = ['2010','2012','2013','2014','2015','2016','2018','2020','2022','2024','2026']

def parse_comm_year(raw):
    """
    Given anything from the Commissioning Year cell:
    1) extract all 4-digit numbers.
    2) FILTER them: keep only plausible years (e.g. 2000 < year < 2060).
    3) if none remain (because they were 0000 or 9999), return NA.
    4) else take the LATEST one (max).
    """
    if pd.isna(raw):
        return pd.NA
    
    s = str(raw)

    candidates = re.findall(r'\b(\d{4})\b', s)
    
    if not candidates:
        return pd.NA
    
    valid_years = []
    for y in candidates:
        y_int = int(y)
        if 2000 <= y_int <= 2060:
            valid_years.append(y_int)
    if not valid_years:
        return pd.NA
        
    return int(max(valid_years))

for year in years_all:
    src_col = (year, 'Inv_Commissioning_Year')
    std_col = (year, 'Inv_Commissioning_Year_Standardized')
    if src_col in merged_2010_2024.columns:
        merged_2010_2024[std_col] = merged_2010_2024[src_col].apply(parse_comm_year)

### Add column 'Inv_Is_Present' to track precence

In [26]:
key_fields = ['Inv_index', 'Inv_Status', 'Project ID', 'Inv_Status_Final']

years_all = ['2010','2012','2013','2014','2015','2016','2018','2020','2022','2024','2026']

for year in years_all:
    available_cols = [ (year, f) for f in key_fields if (year, f) in merged_2010_2024.columns ]
    
    if available_cols:
        merged_2010_2024[(year, 'Inv_Is_Present')] = merged_2010_2024[available_cols].notna().any(axis=1).astype(int)
    else:
        all_year_cols = [c for c in merged_2010_2024.columns if c[0] == year and c[1] != 'Inv_Is_Present']
        
        if all_year_cols:
            merged_2010_2024[(year, 'Inv_Is_Present')] = merged_2010_2024[all_year_cols].notna().any(axis=1).astype(int)
        else:
            merged_2010_2024[(year, 'Inv_Is_Present')] = 0



In [27]:
years_all = ['2010','2012','2013','2014','2015','2016','2018','2020','2022','2024','2026']

key_fields = ['Inv_index', 'Inv_Status', 'Project ID', 'Inv_Status_Final']

for year in years_all:
    available_cols = [(year, f) for f in key_fields if (year, f) in merged_2010_2024.columns]
    
    if available_cols:
        merged_2010_2024[(year, 'Inv_Is_Present')] = (
            merged_2010_2024[available_cols].notna().any(axis=1).astype(int)
        )
    else:
        all_year_cols = [c for c in merged_2010_2024.columns if c[0] == year and c[1] != 'Inv_Is_Present']
        if all_year_cols:
            merged_2010_2024[(year, 'Inv_Is_Present')] = (
                merged_2010_2024[all_year_cols].notna().any(axis=1).astype(int)
            )
        else:
            merged_2010_2024[(year, 'Inv_Is_Present')] = 0

year_masks = {}
for y in years_all:
    col_name = (y, 'Inv_Is_Present')
    if col_name in merged_2010_2024.columns:
        year_masks[y] = merged_2010_2024[col_name] == 1

idx = merged_2010_2024.index
years_present_meta = []

for i in idx:
    active_years = [int(y) for y in years_all if y in year_masks and year_masks[y].loc[i]]
    years_present_meta.append(active_years)

merged_2010_2024[('meta', 'Inv_Years_present')] = years_present_meta
merged_2010_2024[('meta', 'Inv_First_year_present')] = [ys[0] if ys else None for ys in years_present_meta]
merged_2010_2024[('meta', 'Inv_Last_year_present')] = [ys[-1] if ys else None for ys in years_present_meta]



In [28]:
merged_2010_2024 = merged_2010_2024.sort_index(axis=1, level=[0, 1])

### Add Statud: Cancelled/Commissioned


In [29]:
df_copy = merged_2010_2024.copy()

### Reporting error, in 2020-2026, project labeled as "completed" are categorized ad 6, instead of 5

In [30]:
years_to_fix = ['2020', '2022', '2024']

for year in years_to_fix:
    if (year, 'Inv_Status') in df_copy.columns:
        mask_completed = df_copy[(year, 'Inv_Status')] == 6
        df_copy.loc[mask_completed, (year, 'Inv_Status')] = 5
        corretti = mask_completed.sum()

In [31]:
report_years = [2014, 2015, 2016, 2018, 2020, 2022, 2024, 2026]

for year in report_years:
    df_copy[(str(year), "Inv_Status_Derived")] = None

for idx, row in df_copy.iterrows():
    last_year_val = row.get(("meta", "Inv_Last_year_present"))
    
    if pd.isna(last_year_val) or int(last_year_val) not in report_years:
        continue

    last_year = int(last_year_val)

    try:
        status_last_year = row[(str(last_year), "Inv_Status")]
    except KeyError:
        status_last_year = np.nan

    current_status_code = None
    if pd.notna(status_last_year):
        try:
            current_status_code = int(status_last_year)
        except ValueError:
            pass 

    if current_status_code == 5:
        df_copy.at[idx, (str(last_year), "Inv_Status_Derived")] = "Commissioned"
        continue

    elif current_status_code == 6:
        df_copy.at[idx, (str(last_year), "Inv_Status_Derived")] = "Cancelled"
        continue 

    
    # --- INFERENCE LOGIC (If status was not 5 or 6) ---
    # If we reach this point, the project disappeared but the last status was 'Planned', 'Permitting', etc.
    # We infer the outcome based on dates and write it in the FOLLOWING report year.

    try:
        i = report_years.index(last_year)
        next_year = report_years[i + 1]
    except (ValueError, IndexError):
        continue  

    try:
        comm_year = row[(str(last_year), "Inv_Commissioning_Year_Standardized")]
    except KeyError:
        continue

    if pd.isna(comm_year):
        continue

    try:
        comm_year = int(comm_year)
    except ValueError:
        continue


    derived_status = "Cancelled" if comm_year > next_year else "Commissioned"

    try:
        df_copy.at[idx, (str(next_year), "Inv_Status_Derived")] = derived_status
    except KeyError:
        continue


In [32]:
progress_cols = [(str(y), 'Inv_Status_Derived') for y in report_years if (str(y), 'Inv_Status_Derived') in df_copy.columns]

def mark_investment_conclusion(row):
    for col in progress_cols:
        val = row[col]
        if pd.notna(val):
            sval = str(val).strip().lower()
            if "cancelled" in sval:
                return "Cancelled"
            if "commissioned" in sval:
                return "Commissioned"
    return pd.NA

df_copy[('meta', 'Investment Conclusion')] = df_copy.apply(mark_investment_conclusion, axis=1)


In [33]:
cancelled_count = (df_copy[('meta', 'Investment Conclusion')] == "Cancelled").sum()
commissioned_count = (df_copy[('meta', 'Investment Conclusion')] == "Commissioned").sum()
nan_count = df_copy[('meta', 'Investment Conclusion')].isna().sum()

print("Number of Cancelled Investments:", cancelled_count)
print("Number of Commissioned Investments:", commissioned_count)
print("Number of Investments with NaN (no conclusion):", nan_count)

Number of Cancelled Investments: 447
Number of Commissioned Investments: 181
Number of Investments with NaN (no conclusion): 337


In [34]:
report_years_str = [str(y) for y in report_years]

for year in report_years_str:
    if (year, 'Inv_Status') in df_copy.columns:
        df_copy[(year, 'Inv_Status_Final')] = df_copy[(year, 'Inv_Status')].copy()
    else:
        df_copy[(year, 'Inv_Status_Final')] = np.nan

    if (year, 'Inv_Status_Derived') in df_copy.columns:
        mask_comm = df_copy[(year, 'Inv_Status_Derived')] == "Commissioned"
        df_copy.loc[mask_comm, (year, 'Inv_Status_Final')] = 5
        
        mask_canc = df_copy[(year, 'Inv_Status_Derived')] == "Cancelled"
        df_copy.loc[mask_canc, (year, 'Inv_Status_Final')] = 6


In [35]:
report_years_str = [str(y) for y in report_years]

for year in report_years_str:
    
    if (year, 'Inv_Progress') in df_copy.columns:
        df_copy[(year, 'Inv_Progress_Final')] = df_copy[(year, 'Inv_Progress')].astype(object).copy()
    else:
        df_copy[(year, 'Inv_Progress_Final')] = None

    if (year, 'Inv_Status_Derived') in df_copy.columns:
        
        mask_comm = df_copy[(year, 'Inv_Status_Derived')] == "Commissioned"
        df_copy.loc[mask_comm, (year, 'Inv_Progress_Final')] = "Commissioned"
        
        mask_canc = df_copy[(year, 'Inv_Status_Derived')] == "Cancelled"
        df_copy.loc[mask_canc, (year, 'Inv_Progress_Final')] = "Cancelled"

all_values = []
for year in report_years_str:
    if (year, 'Inv_Progress_Final') in df_copy.columns:
        vals = df_copy[(year, 'Inv_Progress_Final')].dropna().unique().tolist()
        all_values.extend(vals)


In [36]:
years_from_2014 = ['2014', '2015', '2016', '2018', '2020', '2022', '2024', '2026']

for year in years_from_2014:
    if (year, 'Inv_Status_Final') in df_copy.columns:
        df_copy[(year, 'Inv_Status')] = df_copy[(year, 'Inv_Status_Final')]
        
    if (year, 'Inv_Progress_Final') in df_copy.columns:
        df_copy[(year, 'Inv_Progress')] = df_copy[(year, 'Inv_Progress_Final')]

cols_to_drop = []
for y in years_from_2014:
    for c in ['Inv_Status_Final', 'Inv_Progress_Final', 'Inv_Status_Derived']:
        if (y, c) in df_copy.columns:
            cols_to_drop.append((y, c))

if cols_to_drop:
    df_copy = df_copy.drop(columns=cols_to_drop, errors='ignore')


In [37]:
meta_cols = [c for c in df_copy.columns if c[0] == 'meta']

target_variables = [
    'Project_ID', 'Inv_index', 'Project_Name', 'Inv_Name', 'Inv_Description',
    'Inv_Status', 'Inv_Progress', 'Inv_Progress Driver', 
    'Inv_Commissioning_Year', 'Inv_Commissioning_Year_Standardized', 'Inv_Is_Present',
    'Project_Promoter', 'Project_PCI_number', 'Inv_Is_main_investment',
    'Project_Country', 'Project_Is_cross_border', 'Inv_Substation From', 'Inv_Substation To',
    'Inv_Element type', 'Inv_Technology[AC/DC]', 'Inv_Capacity [MW]', 'Inv_Voltage [kV]', 'Inv_Line length [km]',
    'Inv_type [New-Upgrade]',
    'Project_type [New-Upgrade]',
]

ordered_cols_multiindex = []

ordered_cols_multiindex.extend(meta_cols)

years_present = [y for y in df_copy.columns.get_level_values(0).unique() if y != 'meta']
for year in sorted(years_present):
    for var in target_variables:
        if (year, var) in df_copy.columns:
            ordered_cols_multiindex.append((year, var))

df_copy = df_copy[ordered_cols_multiindex]


In [38]:
df_copy.to_excel("Results/3.xlsx")
